<a href="https://colab.research.google.com/github/dtrieu9/AI-in-Bio-2/blob/main/pdb_classifier_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein Classifier from the Protein Data Bank

This notebook downloads protein data from the RCSB Protein Data Bank, builds a simple dataset, and trains a multiclass classifier.

We are looking at hemoglobin, lysozyme, insulin, and histone, but you can edit `CLASS_TERMS` to search for other proteins, too.

## 1) Imports and Helper Functions

This code cell contains many functions which will be used later in the notebook.

In [1]:
!wget "https://raw.githubusercontent.com/dtrieu9/AI-in-Bio-2/refs/heads/main/pdb_helpers2.py"

--2026-09-29 19:20:53--  https://raw.githubusercontent.com/dtrieu9/AI-in-Bio-2/refs/heads/main/pdb_helpers2.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12895 (13K) [text/plain]
Saving to: ‘pdb_helpers2.py’

pdb_helpers2.py     100%[===================>]  12.59K  --.-KB/s    in 0s      

2026-09-29 19:20:53 (92.7 MB/s) - ‘pdb_helpers2.py’ saved [12895/12895]



In [2]:
!pip install -q py3Dmol
import pdb_helpers2 as pdb

## 2) Configuration

This cell is for determining which website to use and what proteins to look for.

In [3]:
SEARCH_URL = "https://search.rcsb.org/rcsbsearch/v2/query"
DATA_BASE = "https://data.rcsb.org/rest/v1/core"

# Edit these class buckets as needed.
CLASS_TERMS = {
    "hemoglobin" : "hemoglobin",
    "lysozyme" : "lysozyme",
    "insulin" : "insulin",
    "histone" : "histone"
}

AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")

## 3) Build the dataset from PDB

This may take a few minutes depending on `samples_per_class` and network speed.

In [4]:
# Adjust these if you want a bigger or smaller dataset.
df = pdb.build_or_load_dataset(
    class_terms=CLASS_TERMS,
    samples_per_class=120,
    min_seq_len=40,
)

print("Dataset size:", len(df))
display(df.groupby("label").size().reset_index(name="count"))
display(df.head())

Retrieving data from RCSB PDB...
Searching class 'hemoglobin' with term 'hemoglobin' ...
  collected 20/120 for hemoglobin
  collected 40/120 for hemoglobin
  collected 60/120 for hemoglobin
  collected 80/120 for hemoglobin
  collected 100/120 for hemoglobin
  collected 120/120 for hemoglobin
Collected 120 sequences for 'hemoglobin'.
Searching class 'lysozyme' with term 'lysozyme' ...
  collected 20/120 for lysozyme
  collected 40/120 for lysozyme
  collected 60/120 for lysozyme
  collected 80/120 for lysozyme
  collected 100/120 for lysozyme
  collected 120/120 for lysozyme
Collected 120 sequences for 'lysozyme'.
Searching class 'insulin' with term 'insulin' ...
  collected 20/120 for insulin
  collected 40/120 for insulin
Network error (('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))). Retrying in 1 second(s)...
  collected 60/120 for insulin
  collected 80/120 for insulin
  collected 100/120 for insulin
  collected 120/120 for insulin
Co

,label,count
0,hemoglobin,120
1,histone,120
2,insulin,120
3,lysozyme,120


,entity_id,entry_id,label,sequence,length
0,2PGH_1,2PGH,hemoglobin,VLSAADKANVKAAWGKVGGQAGAHGAEALERMFLGFPTTKTYFPHF...,141
1,2PGH_2,2PGH,hemoglobin,VHLSAEEKEAVLGLWGKVNVDEVGGEALGRLLVVYPWTQRFFESFG...,146
2,3PEL_1,3PEL,hemoglobin,VLSPADKTNIKSTWDKIGGHAGDYGGEALDRTFQSFPTTKTYFPHF...,141
3,3PEL_2,3PEL,hemoglobin,VHLTAEEKSLVSGLWGKVNVDEVGGEALGRLLIVYPWTQRFFDSFG...,146
4,3GOU_1,3GOU,hemoglobin,VLSPADKTNIKSTWDKIGGHAGDYGGEALDRTFQSFPTTKTYFPHF...,141


## 4) Visualization

We are using the `protein_visualization()` to randomly visualize one of the proteins we collected from the PDB. The next cell after that will show us amino acid chain sequences for each protein we collected.

In [5]:
pdb.protein_visualization(df)

Attempting to visualize protein: hemoglobin (Entry ID: 1HBR)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [6]:
with pdb.pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    formatted_output = [f"{index} {sequence}" for index, sequence in df['sequence'].items()]
    print('\n'.join(formatted_output))

0 VLSAADKANVKAAWGKVGGQAGAHGAEALERMFLGFPTTKTYFPHFNLSHGSDQVKAHGQKVADALTKAVGHLDDLPGALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHHPDDFNPSVHASLDKFLANVSTVLTSKYR
1 VHLSAEEKEAVLGLWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSNADAVMGNPKVKAHGKKVLQSFSDGLKHLDNLKGTFAKLSELHCDQLHVDPENFRLLGNVIVVVLARRLGHDFNPDVQAAFQKVVAGVANALAHKYH
2 VLSPADKTNIKSTWDKIGGHAGDYGGEALDRTFQSFPTTKTYFPHFDLSPGSAQVKAHGKKVADALTTAVAHLDDLPGALSALSDLHAYKLRVDPVNFKLLSHCLLVTLACHHPTEFTPAVHASLDKFFAAVSTVLTSKYR
3 VHLTAEEKSLVSGLWGKVNVDEVGGEALGRLLIVYPWTQRFFDSFGDLSTPDAVMSNAKVKAHGKKVLNSFSDGLKNLDNLKGTFAKLSELHCDKLHVDPENFKLLGNVLVCVLAHHFGKEFTPQVQAAYQKVVAGVANALAHKYH
4 VLSPADKTNIKSTWDKIGGHAGDYGGEALDRTFQSFPTTKTYFPHFDLSPGSAQVKAHGKKVADALTTAVAHLDDLPGALSALSDLHAYKLRVDPVNFKLLSHCLLVTLACHHPTEFTPAVHASLDKFFAAVSTVLTSKYR
5 VHLTAEEKSLVSGLWGKVNVDEVGGEALGRLLIVYPWTQRFFDSFGDLSTPDAVMSNAKVKAHGKKVLNSFSDGLKNLDNLKGTFAKLSELHCDKLHVDPENFKLLGNVLVCVLAHHFGKEFTPQVQAAYQKVVAGVANALAHKYH
6 VLSAADKGNVKAAWGKVGGHAAEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGAKVAAALTKAVEHLDDLPGALSELSDLHAHKLRVDPVNFKLLSHSLLVTLASHLPSDFTP

## 5) Train and evaluate

There are many algorithms we could use to create a model that can classify proteins. Here, we apply our data to a Logistic Regression algorithm to create the model. Then we evaluate the model's performance.

In [7]:
model = pdb.train_and_evaluate(df)

Accuracy: 0.9896

Classification report:

              precision    recall  f1-score   support

  hemoglobin       1.00      1.00      1.00        24
     histone       1.00      1.00      1.00        24
     insulin       0.96      1.00      0.98        24
    lysozyme       1.00      0.96      0.98        24

    accuracy                           0.99        96
   macro avg       0.99      0.99      0.99        96
weighted avg       0.99      0.99      0.99        96



## 6) Save the trained model

In machine learning projects, training a model can be costly. You don't want to retrain the model every time you need to use it, so it's good practice to save it for later.

In [8]:
model_path = "protein_classifier.joblib"
pdb.joblib.dump(
    {
        "model": model,
        "class_terms": CLASS_TERMS,
        "aa_order": AA_ORDER,
    },
    model_path,
)
print(f"Saved model to: {model_path}")

Saved model to: protein_classifier.joblib


## 7) Predict on a new sequence

Let's see if the classifier can predict a protein you find yourself on the  PDB. Replace `example_sequence` with any protein sequence in one-letter amino-acid code.

1. Go to [rcsb.org](https://www.rcsb.org/)
2. Search for hemoglobin, lysozyme, insulin, or histone.
3. Click on one of the results
4. Download the FASTA sequence
5. Open the file in Notepad and copy one of the amino acid sequences.
6. Paste the sequence below where it says "example_sequence"



In [9]:
example_sequence = "VLSGEDKSNIKAAWGKIGGHGAEYGAEALERMFASFPTTKTYFPHFDVSHGSAQVKGHGKKVADALASAAGHLDDLPGALSALSDLHAHKLRVDPVNFKLLSHCLLVTLASHHPADFTPAVHASLDKFLASVSTVLTSKYR"
print("Prediction:", model.predict([example_sequence])[0])
print("Probabilities:")
probabilities = dict(zip(model.classes_, model.predict_proba([example_sequence])[0]))
for protein, prob in probabilities.items():
    print(f"{protein}: {prob * 100:.2f}%")

Prediction: hemoglobin
Probabilities:
hemoglobin: 99.96%
histone: 0.04%
insulin: 0.00%
lysozyme: 0.01%
